# Mid-training eval — load LoRA adapter from HF Hub + run 30-prompt eval

Compares against 7B baseline (27/30 syntactic, ~2/30 letter-shaped via `<text>`).

**Edit `ADAPTER_REPO`** below to the repo you pushed from the pod (e.g. `junho5400/svg-finetune-qwen-7b-lora-step2500`).

Runtime → A100, then Run all. ~10-15 min on A100.

In [ ]:
ADAPTER_REPO = 'junho5400/svg-finetune-qwen-7b-lora-step2500'   # CHANGE THIS

!pip install -q 'transformers>=4.45,<4.50' 'peft>=0.13,<0.14' 'torch>=2.1' 'accelerate>=1.0' 'cairosvg>=2.7'

In [ ]:
import json, sys, xml.etree.ElementTree as ET
from pathlib import Path
import torch
import cairosvg
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B'
OUT_DIR = Path('/content/mideval')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_NEW_TOKENS = 1024   # match training max_length
INSTRUCTION_PREFIX = 'Generate an SVG glyph for: '
RESPONSE_TEMPLATE = '\nSVG:\n'

EVAL_PROMPTS = [
    # In-distribution (15)
    ("the letter 'A' in a sans-serif typeface; humanist, calm, easy to read", 'A', 'in-dist'),
    ("the letter 'g' in a sans-serif typeface; geometric, neo-grotesque", 'g', 'in-dist'),
    ("the letter 'h' in a sans-serif typeface; rounded, friendly", 'h', 'in-dist'),
    ("the letter 'B' in a serif typeface; transitional, formal", 'B', 'in-dist'),
    ("the letter 'q' in a serif typeface; old-style, vintage", 'q', 'in-dist'),
    ("the letter 'R' in a serif typeface; modern, high-contrast", 'R', 'in-dist'),
    ("the letter 'M' in a display typeface; playful, rounded, blobby", 'M', 'in-dist'),
    ("the letter '7' in a display typeface; vintage, fat-face", '7', 'in-dist'),
    ("the letter 'W' in a display typeface; futuristic, geometric", 'W', 'in-dist'),
    ("the letter 'p' in a handwriting typeface; informal, cursive", 'p', 'in-dist'),
    ("the letter 'L' in a handwriting typeface; calligraphic, formal", 'L', 'in-dist'),
    ("the letter 'j' in a handwriting typeface; casual, script", 'j', 'in-dist'),
    ("the letter 'X' in a monospace typeface; technical, clean", 'X', 'in-dist'),
    ("the letter '2' in a monospace typeface; geometric, neo-grotesque", '2', 'in-dist'),
    ("the letter 'k' in a monospace typeface; technical, programmer-style", 'k', 'in-dist'),
    # Natural (10)
    ("warm humanist 'C', calm and easy to read", 'C', 'natural'),
    ("geometric 'd' in modern style", 'd', 'natural'),
    ("formal transitional 'E', classical and elegant", 'E', 'natural'),
    ("old-style vintage 's', traditional character", 's', 'natural'),
    ("playful 'N', blobby and bouncy", 'N', 'natural'),
    ("vintage decorative '5', ornate and bold", '5', 'natural'),
    ("flowing 'y' in calligraphic style", 'y', 'natural'),
    ("hand-drawn casual 'T', informal handwriting", 'T', 'natural'),
    ("technical clean '?', monospaced precision", '?', 'natural'),
    ("sharp futuristic '&', geometric", '&', 'natural'),
    # Name references (5)
    ("the letter 'h' in Roboto style", 'h', 'name'),
    ("the letter 'S' in Pacifico style", 'S', 'name'),
    ("the letter 'A' in Comic Sans style", 'A', 'name'),
    ("the letter 'O' in Helvetica style", 'O', 'name'),
    ("the letter 'r' in Times New Roman style", 'r', 'name'),
]
print(f'{len(EVAL_PROMPTS)} prompts loaded')

In [ ]:
print(f'Loading base {BASE_MODEL}...')
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto',
)
print(f'Loading adapter from {ADAPTER_REPO}...')
model = PeftModel.from_pretrained(base, ADAPTER_REPO)
model.eval()
print(f'Loaded. Device: {model.device}, dtype: {model.dtype}')

In [ ]:
def extract_svg(text):
    s = text.find('<svg')
    if s < 0: return None
    e = text.find('</svg>', s)
    if e < 0: return None
    return text[s:e + len('</svg>')]

def is_valid_xml(s):
    try:
        ET.fromstring(s); return True
    except ET.ParseError:
        return False

def uses_text_shortcut(svg):
    return svg is not None and '<text' in svg

In [ ]:
results = []
for i, (caption, target, ptype) in enumerate(EVAL_PROMPTS):
    prompt = f'{INSTRUCTION_PREFIX}{caption}{RESPONSE_TEMPLATE}'
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=tok.pad_token_id,
        )
    new = out_ids[0][inputs.input_ids.shape[1]:]
    gen = tok.decode(new, skip_special_tokens=True)
    svg = extract_svg(gen)
    valid_xml = bool(svg) and is_valid_xml(svg)
    used_text = uses_text_shortcut(svg)
    rendered = False
    (OUT_DIR / f'sample_{i:02d}_raw.txt').write_text(gen)
    if svg:
        (OUT_DIR / f'sample_{i:02d}.svg').write_text(svg)
        try:
            cairosvg.svg2png(bytestring=svg.encode(),
                             write_to=str(OUT_DIR / f'sample_{i:02d}.png'),
                             output_width=128, output_height=128)
            rendered = True
        except Exception:
            pass
    flags = '+'.join(x for x, ok in [('svg', svg), ('xml', valid_xml), ('png', rendered)] if ok) or '—'
    text_flag = ' <text>' if used_text else ''
    print(f'  [{i+1:02d}/{len(EVAL_PROMPTS)}] {target!r:>4}  {ptype:>8}  ({flags:>11}){text_flag}  gen_tokens={int(new.shape[0]):>4}  prompt={caption[:55]}...', flush=True)
    results.append({
        'i': i, 'caption': caption, 'target': target, 'type': ptype,
        'n_gen_tokens': int(new.shape[0]),
        'svg_extracted': svg is not None,
        'valid_xml': valid_xml,
        'used_text': used_text,
        'rendered': rendered,
    })
(OUT_DIR / '_summary.json').write_text(json.dumps(results, indent=2))

In [ ]:
n = len(results)
print(f'=== Summary ({ADAPTER_REPO}) ===')
print(f'  total prompts            : {n}')
print(f'  extracted <svg>          : {sum(r["svg_extracted"] for r in results)}/{n}')
print(f'  valid XML                : {sum(r["valid_xml"] for r in results)}/{n}')
print(f'  rendered to PNG          : {sum(r["rendered"] for r in results)}/{n}')
print(f'  used <text> shortcut     : {sum(r["used_text"] for r in results)}/{n}')
print()
print('  rendered by type:')
for ptype in ['in-dist', 'natural', 'name']:
    rs = [r for r in results if r['type'] == ptype]
    n_rend = sum(r['rendered'] for r in rs)
    n_text = sum(r['used_text'] for r in rs)
    print(f'    {ptype:>8} : {n_rend}/{len(rs)} rendered ({n_text} via <text>)')
print()
print('Compare against 7B baseline (no training): 27/30 rendered, 2 via <text>, ~0 letter-shaped paths.')

from IPython.display import display, Image, Markdown
display(Markdown('## Rendered samples'))
for i in range(n):
    p = OUT_DIR / f'sample_{i:02d}.png'
    r = results[i]
    if p.exists():
        text_note = ' (via `<text>` shortcut)' if r['used_text'] else ''
        display(Markdown(f'**[{i:02d}] {r["type"]} | target=`{r["target"]}`**{text_note} — {r["caption"][:80]}'))
        display(Image(filename=str(p)))

In [ ]:
# Download all results as a zip
import shutil
from google.colab import files
shutil.make_archive('/content/mideval', 'zip', '/content/mideval')
files.download('/content/mideval.zip')